# CNN Assignment: Vegetable Image Classification Using Deep Learning

**Dataset:** Kaggle Vegetable Image Dataset (15 classes)  
**Framework:** TensorFlow / Keras (`Sequential` API)  
**Constraint:** The CNN is built from scratch. No pretrained model and no transfer learning are used.

**Notebook outline**

1. Import required libraries
2. Define dataset paths
3. Count images in each class
4. Create summary DataFrames
5. Visualize class distribution
6. Image preprocessing and augmentation
7. Load images with `flow_from_directory()`
8. Build the CNN model from scratch
9. Compile the model
10. Train the model
11. Evaluate the model
12. Plot accuracy and loss curves

> **Note:** All numbers (image counts, accuracy, loss) are produced by running the code. Nothing is typed in by hand.

## Part 1 – Import Required Libraries

We import libraries for five purposes:

- **Data handling:** `numpy`, `pandas`
- **File path handling:** `os`, `pathlib`
- **Visualization:** `matplotlib`
- **Deep learning:** `tensorflow` and `keras` (model, layers, callbacks)
- **Image preprocessing:** `ImageDataGenerator` (rescaling and augmentation) and `PIL` (to open sample images)

We also fix the random seed (42) so that the results are more reproducible.

In [ ]:
# ---- File path handling ----
import os
from pathlib import Path

# ---- Data handling ----
import numpy as np
import pandas as pd

# ---- Visualization ----
import matplotlib.pyplot as plt

# ---- Image handling ----
from PIL import Image

# ---- Deep learning (TensorFlow / Keras) ----
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# ---- Image preprocessing and augmentation ----
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## Part 2 – Define Dataset Paths

The dataset has three sub-folders: `train`, `validation`, and `test`.  
Each one contains 15 class folders (one folder per vegetable).  
We store the three paths in variables so we can reuse them everywhere in the notebook.

In [ ]:
# Root folder of the dataset (exact Kaggle path)
base_dir = "/kaggle/input/vegetable-image-dataset/Vegetable Images/"

# The three dataset splits
train_dir      = os.path.join(base_dir, "train")
test_dir       = os.path.join(base_dir, "test")
validation_dir = os.path.join(base_dir, "validation")

print("train_dir      :", train_dir)
print("test_dir       :", test_dir)
print("validation_dir :", validation_dir)

# Safety check: stop early with a clear message if a folder is missing
for name, path in [("train", train_dir), ("test", test_dir), ("validation", validation_dir)]:
    if not os.path.isdir(path):
        raise FileNotFoundError(
            f"The '{name}' folder was not found at: {path}\n"
            "Please add the dataset 'misrakahmed/vegetable-image-dataset' to this notebook "
            "using 'Add Input' in Kaggle."
        )
print("\nAll three dataset folders were found.")

### Explore the dataset structure

Before counting, we look at how the dataset is organized: which splits exist, how many class folders each one has, and a few example file names.

In [ ]:
print("Dataset root:", base_dir)
print("Items in root:", sorted(os.listdir(base_dir)), "\n")

for split_name, split_path in [("train", train_dir), ("validation", validation_dir), ("test", test_dir)]:
    class_folders = sorted(d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d)))
    print(f"{split_name}/  ->  {len(class_folders)} class folders")
    print("   ", class_folders)

# Example: first 3 files inside the first class folder of the train split
first_class = sorted(os.listdir(train_dir))[0]
print(f"\nExample files in train/{first_class}/:")
for fname in sorted(os.listdir(os.path.join(train_dir, first_class)))[:3]:
    print("   ", fname)

## Part 3 – Count Images in Each Class

We write one **reusable function**, `count_images_per_class(folder_path)`. It:

1. takes a folder path,
2. reads every class sub-folder,
3. counts the image files inside each class folder (only image extensions are counted),
4. returns a dictionary `{class_name: number_of_images}`.

Then we apply it to the train, test, and validation folders.

In [ ]:
# File extensions that we accept as images
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

def count_images_per_class(folder_path):
    """
    Count the images inside every class folder of `folder_path`.

    Parameters
    ----------
    folder_path : str
        Path of a split folder (train, test, or validation).
        It must contain one sub-folder per class.

    Returns
    -------
    dict
        Dictionary {class_name: number_of_images}, sorted by class name.
    """
    counts = {}
    for class_name in sorted(os.listdir(folder_path)):
        class_path = os.path.join(folder_path, class_name)
        if os.path.isdir(class_path):               # ignore stray files
            n_images = sum(
                1 for f in os.listdir(class_path)
                if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS
            )
            counts[class_name] = n_images
    return counts

# Apply the function to the three splits
train_counts      = count_images_per_class(train_dir)
test_counts       = count_images_per_class(test_dir)
validation_counts = count_images_per_class(validation_dir)

def show_counts(title, counts):
    print(f"=== {title} ===")
    for cls, n in counts.items():
        print(f"{cls:<15}: {n}")
    print(f"{'TOTAL':<15}: {sum(counts.values())}\n")

show_counts("Train image counts", train_counts)
show_counts("Test image counts", test_counts)
show_counts("Validation image counts", validation_counts)

print("Number of classes found:", len(train_counts))

### Sample images (one per class)

A quick visual check: one random training image from each of the 15 classes, with its size.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for ax, cls in zip(axes.ravel(), train_counts.keys()):
    cls_path = os.path.join(train_dir, cls)
    files = [f for f in os.listdir(cls_path) if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS]
    img = Image.open(os.path.join(cls_path, random.choice(files)))
    ax.imshow(img)
    ax.set_title(f"{cls}\n{img.size[0]}x{img.size[1]}", fontsize=10)
    ax.axis("off")
plt.suptitle("One sample training image per class", fontsize=14)
plt.tight_layout()
plt.show()

## Part 4 – Create Summary DataFrames

We convert each dictionary into its own DataFrame (train, test, validation).  
Then we merge the three DataFrames on the `Vegetable` column to build one final table with the columns:
`Vegetable`, `Train Count`, `Test Count`, `Validation Count`.

In [ ]:
# Three separate DataFrames
train_df = pd.DataFrame(list(train_counts.items()), columns=["Vegetable", "Train Count"])
test_df  = pd.DataFrame(list(test_counts.items()),  columns=["Vegetable", "Test Count"])
val_df   = pd.DataFrame(list(validation_counts.items()), columns=["Vegetable", "Validation Count"])

print("Train Count DataFrame")
display(train_df)
print("Test Count DataFrame")
display(test_df)
print("Validation Count DataFrame")
display(val_df)

# Merge them into one final DataFrame
final_df = (
    train_df
    .merge(test_df, on="Vegetable", how="outer")
    .merge(val_df,  on="Vegetable", how="outer")
)[["Vegetable", "Train Count", "Test Count", "Validation Count"]]

print("Final merged DataFrame")
display(final_df)

print("Totals per split:")
print(final_df[["Train Count", "Test Count", "Validation Count"]].sum())

## Part 5 – Visualize Class Distribution

We draw a **grouped bar chart**. For every vegetable there are three bars: train, test, and validation.  
This lets us compare the number of images per class and per split at a glance.

In [ ]:
x = np.arange(len(final_df))   # one position per vegetable
bar_width = 0.27

plt.figure(figsize=(16, 7))
plt.bar(x - bar_width, final_df["Train Count"],      width=bar_width, label="Train")
plt.bar(x,             final_df["Test Count"],       width=bar_width, label="Test")
plt.bar(x + bar_width, final_df["Validation Count"], width=bar_width, label="Validation")

plt.xticks(x, final_df["Vegetable"], rotation=45, ha="right")   # rotated labels
plt.xlabel("Vegetable")
plt.ylabel("Image Count")
plt.title("Class Distribution of Train, Test, and Validation Sets")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Question: Is the dataset balanced?

A dataset is **balanced** when every class has (almost) the same number of images.  
The next cell checks this using the real counts: for each split it prints the smallest and largest class size.
The answer printed below is generated from these actual numbers.

In [ ]:
print("Class balance check (computed from the actual counts)\n")
all_balanced = True

for split in ["Train Count", "Test Count", "Validation Count"]:
    col = final_df[split]
    min_v, max_v = col.min(), col.max()
    min_cls = final_df.loc[col.idxmin(), "Vegetable"]
    max_cls = final_df.loc[col.idxmax(), "Vegetable"]
    ratio = max_v / min_v if min_v > 0 else float("inf")
    balanced = (min_v == max_v)
    all_balanced = all_balanced and balanced
    print(f"{split:<17}: min = {min_v} ({min_cls}), max = {max_v} ({max_cls}), "
          f"max/min ratio = {ratio:.2f}, mean = {col.mean():.1f}")

print()
if all_balanced:
    print("ANSWER: Yes. The dataset is perfectly balanced. Within each split, every one of the "
          f"{len(final_df)} classes has exactly the same number of images "
          "(see the min and max values above, which are equal).")
else:
    print("ANSWER: The dataset is NOT perfectly balanced. The class sizes differ inside at least "
          "one split (see the min, max, and ratio values above).")

print("\nTrain : Test : Validation split proportions (by total images):")
totals = final_df[["Train Count", "Test Count", "Validation Count"]].sum()
print((totals / totals.sum() * 100).round(1).astype(str) + " %")

**Why balance matters:** with a balanced dataset the model does not become biased toward a frequent class, and plain **accuracy** is a fair metric.
If classes were imbalanced, we would need class weights or other metrics such as precision, recall, and F1-score.

## Part 6 – Image Preprocessing and Augmentation

We use Keras `ImageDataGenerator`.

### Training generator (rescaling + augmentation + validation split)

| Setting | Value | Explanation |
|---|---|---|
| `rescale` | 1/255 | Pixel values are 0–255. Scaling to 0–1 makes training faster and more stable. |
| `rotation_range` | 20 | Random rotation up to ±20°. Vegetables are photographed from different angles. |
| `width_shift_range` | 0.1 | Random horizontal shift up to 10% of the width. The object may not be centered. |
| `height_shift_range` | 0.1 | Random vertical shift up to 10% of the height. |
| `shear_range` | 0.1 | Small slanting of the image (perspective change). |
| `zoom_range` | 0.1 | Random zoom in or out by up to 10%. The object can be closer or farther. |
| `horizontal_flip` | True | A vegetable is still the same vegetable when mirrored. |
| `validation_split` | 0.2 | 20% of the images are reserved for validation (see Part 7). |

The values are **small and moderate** on purpose. Very strong augmentation could distort the vegetable too much and make learning harder.
Augmentation creates new variations of the training images each epoch, so the model sees more diversity. This reduces **overfitting**.

### Validation generator (rescaling + validation split ONLY)

Validation data must represent real, unchanged images. If we augmented them, the validation score would not be a fair measure.
So we apply **only** `rescale` and `validation_split`, with the **same split value (0.2)** as the training generator.

In [ ]:
VALIDATION_SPLIT = 0.2

# Training generator: rescaling + augmentation + validation split
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,          # normalize pixels to [0, 1]
    rotation_range=20,          # random rotation (degrees)
    width_shift_range=0.1,      # random horizontal shift
    height_shift_range=0.1,     # random vertical shift
    shear_range=0.1,            # random shear
    zoom_range=0.1,             # random zoom
    horizontal_flip=True,       # random horizontal flip
    validation_split=VALIDATION_SPLIT
)

# Validation generator: ONLY rescaling + validation split (NO augmentation)
val_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=VALIDATION_SPLIT
)

print("Training generator and validation generator were created.")

## Part 7 – Load Images Using `flow_from_directory()`

### Important: how we handle `validation_split` and the existing `validation` folder

The assignment asks for `validation_split` with `subset="training"` and `subset="validation"`.
That method **splits ONE directory into two parts**. The dataset also already contains a separate `validation` folder.
Mixing the two approaches carelessly could cause **data leakage** (the same image appearing in both training and validation).

Our decision, which keeps every requirement of the assignment:

- Both `train_ds` and `val_ds` read from the **same directory: `train_dir`**.
- `train_ds` takes `subset="training"` (80%) and `val_ds` takes `subset="validation"` (the other 20%).
- Both use the **same `validation_split` (0.2)**. Keras assigns the files to the subsets deterministically, so the two subsets **never overlap**. We prove this with a check below.
- Both use the **same class order** (alphabetical). We check that the class mappings are identical.
- The original `validation` folder is **not mixed into training or into the split**. It stays untouched as an extra independent set. We still counted it in Parts 3–5, and we use it only for an optional extra check in Part 11.
- The `test` folder is never used for training or model selection.

Required settings: `target_size=(224, 224)`, `batch_size=32`, `class_mode="categorical"`, `seed=42`.
For `val_ds` we use `shuffle=False`. This does not change which images belong to the subset. It only keeps the order fixed so evaluation is repeatable.

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# Training subset (80% of train_dir) - with augmentation
train_ds = train_datagen.flow_from_directory(
    directory=train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    seed=42,
    subset="training"
)

# Validation subset (20% of the SAME train_dir) - no augmentation
val_ds = val_datagen.flow_from_directory(
    directory=train_dir,            # same directory => no leakage between the two subsets
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    seed=42,
    subset="validation",
    shuffle=False                   # fixed order for repeatable evaluation
)

### Consistency checks

We verify (1) that the class mappings are identical, and (2) that there is **zero overlap** between the training and validation images.

In [ ]:
# 1) Class mappings must be identical
assert train_ds.class_indices == val_ds.class_indices, "Class mappings are different!"
print("Class mappings are identical in train_ds and val_ds.\n")
print("Class indices:")
for cls, idx in train_ds.class_indices.items():
    print(f"  {idx:>2} -> {cls}")

num_classes = len(train_ds.class_indices)
class_names = list(train_ds.class_indices.keys())
print("\nNumber of classes:", num_classes)

# 2) No image may appear in both subsets (data leakage check)
overlap = set(train_ds.filenames) & set(val_ds.filenames)
print("\nImages in train_ds :", train_ds.samples)
print("Images in val_ds   :", val_ds.samples)
print("Overlapping images :", len(overlap))
assert len(overlap) == 0, "Data leakage detected!"
print("No overlap between training and validation subsets (no data leakage).")

### A quick look at the loaded data

We display a few training images (these are **augmented**) and a few validation images (these are **not augmented**).

In [ ]:
def show_batch(generator, title, n=8):
    images, labels = next(generator)
    plt.figure(figsize=(16, 4))
    for i in range(min(n, len(images))):
        plt.subplot(1, n, i + 1)
        plt.imshow(images[i])
        plt.title(class_names[np.argmax(labels[i])], fontsize=9)
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_batch(train_ds, "Training batch (rescaled + augmented)")
show_batch(val_ds,   "Validation batch (rescaled only)")

# Reset the generators so training starts from the first batch
train_ds.reset()
val_ds.reset()

## Part 8 – Build the CNN Model From Scratch

We use `Sequential` with exactly the requested architecture. No pretrained weights are used.

| Block | Layers |
|---|---|
| Convolution Block 1 | Conv2D(32, (3,3), ReLU, input_shape=(224,224,3)) → MaxPooling2D |
| Convolution Block 2 | Conv2D(64, (3,3), ReLU) → MaxPooling2D |
| Convolution Block 3 | Conv2D(128, (3,3), ReLU) → MaxPooling2D |
| Fully connected | Flatten → Dense(512, ReLU) → Dropout(0.5) → Dense(15, Softmax) |

In [ ]:
model = Sequential([
    # ---- Convolution Block 1 ----
    Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3)),
    MaxPooling2D(),

    # ---- Convolution Block 2 ----
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(),

    # ---- Convolution Block 3 ----
    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(),

    # ---- Fully connected layers ----
    Flatten(),
    Dense(512, activation="relu"),
    Dropout(0.5),
    Dense(15, activation="softmax")      # 15 vegetable classes
])

print("Model built.")

## Part 9 – Compile the Model

- **Optimizer: Adam.** It adapts the learning rate for each weight and usually converges quickly with little tuning.
- **Loss: Categorical Crossentropy.** It is the standard loss for multi-class classification with one-hot labels (`class_mode="categorical"`).
- **Metric: Accuracy.** The percentage of correctly classified images.

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

### Explanations

**Number of convolutional layers:** The model has **3** convolutional layers (`Conv2D` with 32, 64, and 128 filters). Each one is followed by a pooling layer.
Early layers learn simple features such as edges and colors. Deeper layers combine them into complex features such as textures and shapes.

**Purpose of pooling layers (`MaxPooling2D`):** They reduce the width and height of the feature maps (here by half) by keeping the strongest value in each small window.
This lowers the number of parameters and the computation, makes the model less sensitive to small shifts of the object, and helps reduce overfitting.

**Purpose of dropout (`Dropout(0.5)`):** During training, it randomly switches off 50% of the neurons of the previous layer in each step.
The network cannot depend on specific neurons, so it learns more general features. This is a regularization technique that reduces overfitting. Dropout is turned off automatically during evaluation and prediction.

**Why Softmax in the output layer:** Softmax converts the 15 raw output scores into 15 probabilities that are all between 0 and 1 and **sum to 1**.
This fits a multi-class problem where each image belongs to exactly one class. The predicted class is the one with the highest probability, and categorical crossentropy needs these probabilities.

## Part 10 – Train the Model

- Training data: `train_ds`. Validation data: `val_ds`.
- Maximum **20 epochs**.
- **EarlyStopping** with `monitor="val_loss"`, `patience=5`, `restore_best_weights=True`.

### Why is EarlyStopping useful?

If we train too long, the model starts to memorize the training images. Training loss keeps falling, but validation loss starts to rise. This is **overfitting**.
EarlyStopping watches `val_loss` and stops training when it has not improved for 5 epochs in a row.
With `restore_best_weights=True`, the model returns to the weights of the **best epoch**, not the last one.
It saves training time and gives a model that generalizes better.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

EPOCHS = 20   # maximum number of epochs

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

print("\nEpochs actually trained:", len(history.history["loss"]))
if early_stopping.stopped_epoch > 0:
    print("EarlyStopping stopped training at epoch", early_stopping.stopped_epoch + 1)
else:
    print("EarlyStopping did not trigger; all", EPOCHS, "epochs were completed.")

## Part 11 – Evaluate the Model

We evaluate the final model (with the best weights restored by EarlyStopping) on the training and validation datasets and print the four required values.

**Note about the training score:** `train_ds` applies random augmentation, so evaluating on it measures performance on *augmented* images.
For a cleaner comparison we also evaluate the same 80% training subset **without augmentation** (rescaling only). The required training metrics are the ones computed on `train_ds`; the extra row is shown as a fair reference.

In [ ]:
# Required evaluation on train_ds and val_ds
train_loss, train_acc = model.evaluate(train_ds, verbose=0)
val_loss,   val_acc   = model.evaluate(val_ds,   verbose=0)

print("=== Evaluation Results (train_ds and val_ds) ===")
print(f"Training Accuracy   : {train_acc:.4f}  ({train_acc*100:.2f}%)")
print(f"Validation Accuracy : {val_acc:.4f}  ({val_acc*100:.2f}%)")
print(f"Training Loss       : {train_loss:.4f}")
print(f"Validation Loss     : {val_loss:.4f}")

In [ ]:
# Extra reference: the SAME training subset, but without augmentation
train_eval_ds = val_datagen.flow_from_directory(
    directory=train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    seed=42,
    subset="training",
    shuffle=False
)
assert train_eval_ds.class_indices == train_ds.class_indices

clean_train_loss, clean_train_acc = model.evaluate(train_eval_ds, verbose=0)
print(f"\n[Extra] Training Accuracy without augmentation: {clean_train_acc:.4f}")
print(f"[Extra] Training Loss without augmentation    : {clean_train_loss:.4f}")

In [ ]:
# Optional extra check: the ORIGINAL validation folder (never used in training or in the split)
extra_val_ds = ImageDataGenerator(rescale=1.0 / 255).flow_from_directory(
    directory=validation_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)
assert extra_val_ds.class_indices == train_ds.class_indices

extra_loss, extra_acc = model.evaluate(extra_val_ds, verbose=0)
print(f"\n[Extra] Accuracy on the original 'validation' folder: {extra_acc:.4f} ({extra_acc*100:.2f}%)")
print(f"[Extra] Loss on the original 'validation' folder    : {extra_loss:.4f}")

### Question: Is there a large difference between training accuracy and validation accuracy? What does this tell us about overfitting?

The next cell measures the gap using the real results above and prints the conclusion.

In [ ]:
acc_gap  = train_acc - val_acc
loss_gap = val_loss - train_loss

print(f"Training accuracy   : {train_acc*100:.2f}%")
print(f"Validation accuracy : {val_acc*100:.2f}%")
print(f"Accuracy gap (train - validation): {acc_gap*100:.2f} percentage points")
print(f"Loss gap (validation - train)     : {loss_gap:.4f}\n")

if acc_gap <= 0.03:
    print("ANSWER: No, the difference is small (3 points or less). The model performs almost the same on "
          "training and validation images, so there is no strong sign of overfitting. The model "
          "generalizes well. Data augmentation and dropout probably helped.")
elif acc_gap <= 0.08:
    print("ANSWER: The difference is moderate (between 3 and 8 points). There are mild signs of "
          "overfitting: the model fits the training data somewhat better than unseen data. It is "
          "acceptable, but more regularization or more data could still help.")
else:
    print("ANSWER: Yes, the difference is large (more than 8 points). This is a clear sign of overfitting: "
          "the model memorizes the training images and does not generalize as well to new images. "
          "More dropout, stronger augmentation, or a smaller model could reduce it.")

if acc_gap < 0:
    print("\nNote: validation accuracy is higher than training accuracy. This can happen because the "
          "training score is measured on augmented (harder) images and with dropout active during the "
          "epoch average, while validation images are not augmented.")

## Part 12 – Plot Accuracy and Loss Curves

We create **two separate plots** from the `history` object: (1) training vs. validation accuracy, and (2) training vs. validation loss.
Learning curves show how the model improves over epochs. If training and validation curves move apart, it indicates overfitting.

In [ ]:
epochs_range = range(1, len(history.history["accuracy"]) + 1)

# Plot 1: Training Accuracy vs Validation Accuracy
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, history.history["accuracy"],     marker="o", label="Training Accuracy")
plt.plot(epochs_range, history.history["val_accuracy"], marker="s", label="Validation Accuracy")
plt.title("Training Accuracy vs Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.xticks(list(epochs_range))
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
# Plot 2: Training Loss vs Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, history.history["loss"],     marker="o", label="Training Loss")
plt.plot(epochs_range, history.history["val_loss"], marker="s", label="Validation Loss")
plt.title("Training Loss vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.xticks(list(epochs_range))
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## Conclusion

- The dataset has 15 vegetable classes in three folders (train, test, validation). The class balance was checked from the real counts in Part 5.
- A CNN with 3 convolutional blocks and a fully connected head was built **from scratch** with `Sequential`, without any pretrained model.
- To avoid data leakage, `train_ds` and `val_ds` were both created from `train_dir` with the same `validation_split`, and we verified that they do not overlap.
- Training used Adam, categorical crossentropy, a maximum of 20 epochs, and EarlyStopping (`val_loss`, patience 5, best weights restored).
- The final accuracy and loss values, and the overfitting analysis, come from the actual run in Parts 11 and 12.